In [ ]:
import numpy as np
import pandas as pd
import random

RANDOM_SEED = 86

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# Simulation timeline
# ------------------------------------------------------------

SIMULATION_START = pd.Timestamp("2014-01-01")
SIMULATION_END = pd.Timestamp("2025-12-01")

MONTHS = pd.date_range(start=SIMULATION_START,
    end=SIMULATION_END, freq="MS")

# Market structure
# ------------------------------------------------------------

REGIONS = ["North", "South", "East", "West"]
THERAPIES = ["Therapy A", "Therapy B", "Therapy C", "Therapy D"]
BIOMARKERS = ["Positive", "Negative"]
LINES_OF_THERAPY = ["1L", "2L"]

# Basic contract checks
# ------------------------------------------------------------

assert len(MONTHS) == 144, "The simulation must contain 144 months."
assert len(REGIONS) == 4, "The design requires four regions."
assert len(THERAPIES) == 4, "The design requires four therapies."
assert len(BIOMARKERS) == 2, "The design requires two biomarker groups."
assert len(LINES_OF_THERAPY) == 2, "The design requires two treatment lines."

print("Simulation setup complete")
print(f"Period: {MONTHS.min():%b %Y} to {MONTHS.max():%b %Y}")
print(f"Number of months: {len(MONTHS)}")
print(f"Regions: {REGIONS}")
print(f"Therapies: {THERAPIES}")
print(f"Random seed: {RANDOM_SEED}")

In [ ]:
# ASSUMPTIONS REGISTRY
# ============================================================

WORLD_BANK_REFERENCE = (
    "World Bank, Population growth (annual %), United States, "
    "indicator SP.POP.GROW"
)

GANTI_2021_REFERENCE = (
    "Ganti et al. Update of Incidence, Prevalence, Survival, and "
    "Initial Treatment in Patients With NSCLC in the US, "
    "JAMA Oncology, 2021, PMID: 34673888"
)

SYNTHETIC_REFERENCE = (
    "Assumption; not external evidence"
)


def make_assumption(
    assumption_id,
    assumption_name,
    value,
    unit,
    category,
    source_type,
    source_reference,
    confidence,
    scenario_enabled,
    notes
):
    return {
        "assumption_id": assumption_id,
        "assumption_name": assumption_name,
        "value": value,
        "unit": unit,
        "category": category,
        "source_type": source_type,
        "source_reference": source_reference,
        "confidence": confidence,
        "scenario_enabled": scenario_enabled,
        "notes": notes
    }


assumptions = [
    # --------------------------------------------------------
    # Population
    # --------------------------------------------------------
    make_assumption(
        "SIM_001",
        "Annual population growth rate",
        0.005,
        "proportion",
        "simulation",
        "public_data_informed_proxy",
        WORLD_BANK_REFERENCE,
        "high_for_US_proxy",
        False,
        (
            "Uses 0.5% annual growth as a US-inspired national trend "
            "with simplified assumption of four regional population."
        )
    ),

    # --------------------------------------------------------
    # Epidemiology
    # --------------------------------------------------------
    make_assumption(
        "EPI_001",
        "Annual diagnosed NSCLC incidence rate",
        40.9,
        "cases_per_100k_per_year",
        "epidemiology",
        "evidence_backed",
        GANTI_2021_REFERENCE,
        "high_for_US_2017_estimate",
        True,
        (
            "Published diagnosed NSCLC incidence. This is not latent "
            "disease incidence."
        )
    ),
    make_assumption(
        "EPI_002",
        "Metastatic-stage share among diagnosed NSCLC cases",
        19.6 / 40.9,
        "proportion",
        "epidemiology",
        "evidence_backed_derived",
        GANTI_2021_REFERENCE,
        "high_for_US_2017_estimate",
        True,
        (
            "Derived as stage IV incidence (19.6 per 100k) divided by "
            "overall diagnosed NSCLC incidence (40.9 per 100k)."
        )
    ),
    make_assumption(
        "EPI_003",
        "Diagnosis rate",
        0.80,
        "proportion",
        "epidemiology",
        "synthetic_design",
        SYNTHETIC_REFERENCE,
        "design_choice",
        True,
        (
            "Used to create an unobserved latent disease-incidence layer. "
            "It is calibrated so baseline diagnosed incidence remains 40.9."
        )
    ),
    make_assumption(
        "EPI_004",
        "NSCLC prevalence at simulation start",
        198.3,
        "cases_per_100k",
        "epidemiology",
        "evidence_backed",
        GANTI_2021_REFERENCE,
        "high_for_US_2016_estimate",
        False,
        (
            "All NSCLC prevalence at simulation start. "
            "bridge will identify patients relevant to therapy market."
        )
    ),
    make_assumption(
        "EPI_005",
        "Epidemiology change start month",
        115,
        "month_index",
        "epidemiology",
        "synthetic_design",
        SYNTHETIC_REFERENCE,
        "design_choice",
        True,
        "A pre-specified change in disease incidence begins in month 115."
    ),
    make_assumption(
        "EPI_006",
        "Incidence multiplier after epidemiology change",
        1.12,
        "multiplier",
        "epidemiology",
        "synthetic_design",
        SYNTHETIC_REFERENCE,
        "design_choice",
        True,
        "A 12% increase in latent disease incidence after the regime change."
    ),

    # --------------------------------------------------------
    # Clinical market structure
    # --------------------------------------------------------
    make_assumption(
        "CLN_001",
        "Composite biomarker-positive prevalence",
        0.30,
        "proportion",
        "clinical",
        "synthetic_design",
        SYNTHETIC_REFERENCE,
        "design_choice",
        True,
        (
            "Assumed composite biomarker group used for the synthetic "
            "therapy market; not claimed as a real biomarker prevalence."
        )
    ),
    make_assumption(
        "CLN_002",
        "Monthly first-line initiation rate",
        0.72,
        "proportion",
        "clinical",
        "synthetic_design",
        SYNTHETIC_REFERENCE,
        "design_choice",
        False,
        "Share of eligible first-line candidates initiating treatment."
    ),
    make_assumption(
        "CLN_003",
        "Monthly first-line progression rate",
        0.06,
        "proportion",
        "clinical",
        "synthetic_design",
        SYNTHETIC_REFERENCE,
        "design_choice",
        False,
        "Share of first-line active patients progressing to second-line eligibility."
    ),

    # --------------------------------------------------------
    # Persistence
    # --------------------------------------------------------
    make_assumption(
        "PERS_001", "Therapy A monthly discontinuation rate", 0.080,
        "proportion", "persistence", "synthetic_design",
        SYNTHETIC_REFERENCE, "design_choice", True,
        "Applied to active Therapy A patients."
    ),
    make_assumption(
        "PERS_002", "Therapy B monthly discontinuation rate", 0.075,
        "proportion", "persistence", "synthetic_design",
        SYNTHETIC_REFERENCE, "design_choice", True,
        "Applied to active Therapy B patients."
    ),
    make_assumption(
        "PERS_003", "Therapy C monthly discontinuation rate", 0.085,
        "proportion", "persistence", "synthetic_design",
        SYNTHETIC_REFERENCE, "design_choice", True,
        "Applied to active Therapy C patients."
    ),
    make_assumption(
        "PERS_004", "Therapy D monthly discontinuation rate", 0.065,
        "proportion", "persistence", "synthetic_design",
        SYNTHETIC_REFERENCE, "design_choice", True,
        "Applied after Therapy D launches."
    ),

    # --------------------------------------------------------
    # Demand conversion
    # --------------------------------------------------------
    make_assumption(
        "COMM_001", "Therapy A units per patient-month", 1.00,
        "pack_equivalents_per_patient_month", "commercial",
        "synthetic_design", SYNTHETIC_REFERENCE, "design_choice", False,
        "Synthetic utilization conversion factor."
    ),
    make_assumption(
        "COMM_002", "Therapy B units per patient-month", 1.00,
        "pack_equivalents_per_patient_month", "commercial",
        "synthetic_design", SYNTHETIC_REFERENCE, "design_choice", False,
        "Synthetic utilization conversion factor."
    ),
    make_assumption(
        "COMM_003", "Therapy C units per patient-month", 1.10,
        "pack_equivalents_per_patient_month", "commercial",
        "synthetic_design", SYNTHETIC_REFERENCE, "design_choice", False,
        "Slightly higher synthetic monthly utilization."
    ),
    make_assumption(
        "COMM_004", "Therapy D units per patient-month", 1.00,
        "pack_equivalents_per_patient_month", "commercial",
        "synthetic_design", SYNTHETIC_REFERENCE, "design_choice", False,
        "Synthetic utilization conversion factor."
    ),

    # --------------------------------------------------------
    # Competition
    # --------------------------------------------------------
    make_assumption(
        "COMP_001", "Therapy D launch month number", 85,
        "month_index", "competition", "synthetic_design",
        SYNTHETIC_REFERENCE, "design_choice", True,
        "Therapy D launches in month 85: January 2021."
    )
]

assumption_registry = pd.DataFrame(assumptions)


# ------------------------------------------------------------
# Data-quality checks
# ------------------------------------------------------------

required_columns = {
    "assumption_id", "assumption_name", "value", "unit",
    "category", "source_type", "source_reference",
    "confidence", "scenario_enabled", "notes"
}

assert set(assumption_registry.columns) == required_columns
assert assumption_registry["assumption_id"].is_unique
assert assumption_registry["value"].notna().all()

proportion_rows = assumption_registry["unit"].eq("proportion")
assert assumption_registry.loc[proportion_rows, "value"].between(0, 1).all()

print(f"Assumption registry created: {len(assumption_registry)} assumptions")

display(
    assumption_registry[
        ["assumption_id", "assumption_name", "value", "source_type", "scenario_enabled"]
    ]
)

In [ ]:
# POPULATION AND EPIDEMIOLOGY
# ============================================================

def get_assumption_value(assumption_id):
    """Return one numeric value from the assumptions registry."""

    matching_value = assumption_registry.loc[
        assumption_registry["assumption_id"].eq(assumption_id),
        "value"
    ]

    assert len(matching_value) == 1, (
        f"Expected exactly one value for {assumption_id}"
    )

    return matching_value.iloc[0]


# ------------------------------------------------------------
# Read assumptions
# ------------------------------------------------------------

population_growth_rate = get_assumption_value("SIM_001")

diagnosed_nsclc_incidence_anchor = get_assumption_value("EPI_001")
metastatic_stage_share = get_assumption_value("EPI_002")
base_diagnosis_rate = get_assumption_value("EPI_003")

epi_change_start_month = int(get_assumption_value("EPI_005"))
epi_change_multiplier = get_assumption_value("EPI_006")


# ------------------------------------------------------------
# Derive latent disease incidence
# ------------------------------------------------------------
# We use the published diagnosed incidence as the anchor.
# Diagnosis rate is a synthetic mechanism that sits upstream.
#
# latent incidence × diagnosis rate = diagnosed incidence
# ------------------------------------------------------------

base_latent_disease_incidence = (
    diagnosed_nsclc_incidence_anchor / base_diagnosis_rate
)

assert np.isclose(
    base_latent_disease_incidence * base_diagnosis_rate,
    diagnosed_nsclc_incidence_anchor
)

print(
    "Baseline latent disease incidence: "
    f"{base_latent_disease_incidence:.3f} per 100k per year"
)


# ------------------------------------------------------------
# Define fictional regional variation
# ------------------------------------------------------------

region_profiles = pd.DataFrame([
    {
        "region": "North",
        "starting_population": 11_000_000,
        "latent_incidence_multiplier": 1.10,
        "diagnosis_multiplier": 1.05
    },
    {
        "region": "South",
        "starting_population": 10_000_000,
        "latent_incidence_multiplier": 0.90,
        "diagnosis_multiplier": 0.95
    },
    {
        "region": "East",
        "starting_population": 9_000_000,
        "latent_incidence_multiplier": 1.05,
        "diagnosis_multiplier": 0.90
    },
    {
        "region": "West",
        "starting_population": 10_000_000,
        "latent_incidence_multiplier": 0.95,
        "diagnosis_multiplier": 1.02
    }
])

assert set(region_profiles["region"]) == set(REGIONS)
assert (region_profiles["starting_population"] > 0).all()

regional_diagnosis_rates = (
    base_diagnosis_rate * region_profiles["diagnosis_multiplier"]
)

assert regional_diagnosis_rates.between(0, 1).all()


# ------------------------------------------------------------
# Generate the epidemiology table
# ------------------------------------------------------------

epidemiology_records = []

for region_row in region_profiles.itertuples(index=False):

    for month_index, month in enumerate(MONTHS, start=1):

        years_since_start = (month_index - 1) / 12

        population = (
            region_row.starting_population
            * (1 + population_growth_rate) ** years_since_start
        )

        epidemiology_regime = (
            "epidemiology_change"
            if month_index >= epi_change_start_month
            else "baseline"
        )

        regime_incidence_multiplier = (
            epi_change_multiplier
            if month_index >= epi_change_start_month
            else 1.0
        )

        annual_latent_disease_incidence_per_100k = (
            base_latent_disease_incidence
            * region_row.latent_incidence_multiplier
            * regime_incidence_multiplier
        )

        diagnosis_rate = (
            base_diagnosis_rate
            * region_row.diagnosis_multiplier
        )

        annual_diagnosed_nsclc_incidence_per_100k = (
            annual_latent_disease_incidence_per_100k
            * diagnosis_rate
        )

        incident_disease_cases = (
            population
            * annual_latent_disease_incidence_per_100k
            / 100_000
            / 12
        )

        new_diagnosed_nsclc_cases = (
            incident_disease_cases
            * diagnosis_rate
        )

        new_metastatic_diagnoses = (
            new_diagnosed_nsclc_cases
            * metastatic_stage_share
        )

        epidemiology_records.append({
            "month": month,
            "month_index": month_index,
            "region": region_row.region,
            "population": population,
            "epidemiology_regime": epidemiology_regime,
            "annual_latent_disease_incidence_per_100k_truth":
                annual_latent_disease_incidence_per_100k,
            "diagnosis_rate_truth": diagnosis_rate,
            "annual_diagnosed_nsclc_incidence_per_100k_truth":
                annual_diagnosed_nsclc_incidence_per_100k,
            "metastatic_stage_share_truth": metastatic_stage_share,
            "incident_disease_cases": incident_disease_cases,
            "new_diagnosed_nsclc_cases": new_diagnosed_nsclc_cases,
            "new_metastatic_diagnoses": new_metastatic_diagnoses
        })

epidemiology_monthly = pd.DataFrame(epidemiology_records)


# ------------------------------------------------------------
# Reconciliation and data-quality checks
# ------------------------------------------------------------

expected_rows = len(REGIONS) * len(MONTHS)

assert len(epidemiology_monthly) == expected_rows
assert epidemiology_monthly["month"].nunique() == len(MONTHS)
assert epidemiology_monthly["region"].nunique() == len(REGIONS)

assert (epidemiology_monthly["population"] > 0).all()
assert (epidemiology_monthly["incident_disease_cases"] >= 0).all()
assert (epidemiology_monthly["new_diagnosed_nsclc_cases"] >= 0).all()
assert (epidemiology_monthly["new_metastatic_diagnoses"] >= 0).all()

assert epidemiology_monthly["diagnosis_rate_truth"].between(0, 1).all()
assert epidemiology_monthly["metastatic_stage_share_truth"].between(0, 1).all()

assert np.allclose(
    epidemiology_monthly["new_diagnosed_nsclc_cases"],
    epidemiology_monthly["incident_disease_cases"]
    * epidemiology_monthly["diagnosis_rate_truth"]
)

assert np.allclose(
    epidemiology_monthly["new_metastatic_diagnoses"],
    epidemiology_monthly["new_diagnosed_nsclc_cases"]
    * epidemiology_monthly["metastatic_stage_share_truth"]
)

print(f"Epidemiology table created: {len(epidemiology_monthly):,} rows")

display(
    epidemiology_monthly
    .groupby("region", as_index=False)
    .agg(
        total_incident_disease_cases=("incident_disease_cases", "sum"),
        total_diagnosed_nsclc_cases=("new_diagnosed_nsclc_cases", "sum"),
        total_metastatic_diagnoses=("new_metastatic_diagnoses", "sum")
    )
    .round(1)
)

# patient-pool assumptions
# ------------------------------------------------------------

patient_pool_assumptions = [
    make_assumption(
        "CLN_004",
        "Prevalent NSCLC to active metastatic-treatment pool bridge",
        0.06,
        "proportion",
        "clinical",
        "synthetic_design",
        SYNTHETIC_REFERENCE,
        "design_choice",
        True,
        (
            "Share of all prevalent NSCLC patients represented as active "
            "metastatic patients in the fictional treatment market at Month 1."
        )
    ),
    make_assumption(
        "CLN_005",
        "Initial active patients in first line",
        0.70,
        "proportion",
        "clinical",
        "synthetic_design",
        SYNTHETIC_REFERENCE,
        "design_choice",
        False,
        (
            "At simulation start, 70% of active treated patients are in "
            "first line and 30% are in second line."
        )
    )
]

assumption_registry = pd.concat(
    [assumption_registry, pd.DataFrame(patient_pool_assumptions)],
    ignore_index=True
)

assert assumption_registry["assumption_id"].is_unique


# Assumptions
# ------------------------------------------------------------

biomarker_positive_share = get_assumption_value("CLN_001")
starting_nsclc_prevalence_per_100k = get_assumption_value("EPI_004")
active_metastatic_pool_bridge = get_assumption_value("CLN_004")
initial_first_line_share = get_assumption_value("CLN_005")


# ------------------------------------------------------------
# Create the starting prevalent treatment pool
# ------------------------------------------------------------

starting_population_by_region = (
    epidemiology_monthly
    .query("month_index == 1")
    [["region", "population"]]
    .copy()
)

starting_population_by_region["starting_prevalent_nsclc_cases"] = (
    starting_population_by_region["population"]
    * starting_nsclc_prevalence_per_100k
    / 100_000
)

starting_population_by_region["starting_active_metastatic_patients"] = (
    starting_population_by_region["starting_prevalent_nsclc_cases"]
    * active_metastatic_pool_bridge
)


# ------------------------------------------------------------
# Create all region × biomarker × line × month combinations
# ------------------------------------------------------------

segment_definition = pd.DataFrame(
    [
        {"biomarker_status": biomarker, "line_of_therapy": line}
        for biomarker in BIOMARKERS
        for line in LINES_OF_THERAPY
    ]
)

patient_segment_inputs = epidemiology_monthly.merge(
    segment_definition,
    how="cross"
)

patient_segment_inputs = patient_segment_inputs.merge(
    starting_population_by_region[
        ["region", "starting_active_metastatic_patients"]
    ],
    on="region",
    how="left"
)


# ------------------------------------------------------------
# Split patients by biomarker and line of therapy
# ------------------------------------------------------------

patient_segment_inputs["biomarker_share"] = np.where(
    patient_segment_inputs["biomarker_status"].eq("Positive"),
    biomarker_positive_share,
    1 - biomarker_positive_share
)

patient_segment_inputs["initial_line_share"] = np.where(
    patient_segment_inputs["line_of_therapy"].eq("1L"),
    initial_first_line_share,
    1 - initial_first_line_share
)

# New metastatic diagnoses enter first-line only.
patient_segment_inputs["new_first_line_diagnosis_inflow"] = np.where(
    patient_segment_inputs["line_of_therapy"].eq("1L"),
    patient_segment_inputs["new_metastatic_diagnoses"]
    * patient_segment_inputs["biomarker_share"],
    0.0
)

# Only Month 1 contains the starting prevalent active-treatment pool.
patient_segment_inputs["initial_active_patients"] = np.where(
    patient_segment_inputs["month_index"].eq(1),
    patient_segment_inputs["starting_active_metastatic_patients"]
    * patient_segment_inputs["biomarker_share"]
    * patient_segment_inputs["initial_line_share"],
    0.0
)

# Progression into second line is calculated later, after active patients exist.
patient_segment_inputs["progression_inflow"] = 0.0


# ------------------------------------------------------------
# Data-quality checks
# ------------------------------------------------------------

expected_rows = (
    len(REGIONS)
    * len(BIOMARKERS)
    * len(LINES_OF_THERAPY)
    * len(MONTHS)
)

assert len(patient_segment_inputs) == expected_rows
assert (patient_segment_inputs["new_first_line_diagnosis_inflow"] >= 0).all()
assert (patient_segment_inputs["initial_active_patients"] >= 0).all()

# All new metastatic diagnoses must reconcile to first-line segment inflow.
first_line_reconciliation = (
    patient_segment_inputs
    .groupby(["region", "month", "new_metastatic_diagnoses"], as_index=False)
    .agg(
        total_first_line_inflow=(
            "new_first_line_diagnosis_inflow",
            "sum"
        )
    )
)

assert np.allclose(
    first_line_reconciliation["new_metastatic_diagnoses"],
    first_line_reconciliation["total_first_line_inflow"]
)

# Month-1 segment totals must reconcile to the starting active-treatment pool.
initial_active_reconciliation = (
    patient_segment_inputs
    .query("month_index == 1")
    .groupby("region", as_index=False)
    .agg(
        initial_active_patients=("initial_active_patients", "sum"),
        expected_active_patients=(
            "starting_active_metastatic_patients",
            "first"
        )
    )
)

assert np.allclose(
    initial_active_reconciliation["initial_active_patients"],
    initial_active_reconciliation["expected_active_patients"]
)

print(f"Updated registry: {len(assumption_registry)} assumptions")
print(f"Patient-segment input table created: {len(patient_segment_inputs):,} rows")

display(
    initial_active_reconciliation.round(1)
)

display(
    patient_segment_inputs
    .query("month_index == 1")
    [[
        "region",
        "biomarker_status",
        "line_of_therapy",
        "new_first_line_diagnosis_inflow",
        "initial_active_patients"
    ]]
    .round(1)
)

In [ ]:
# ============================================================
# BLOCK 5 — Therapy Eligibility, Event Calendar, and Market Access
# ============================================================

# Why this block exists:
# We now define which therapies can treat which patient segments,
# when therapies enter the market, which market events occur,
# and what access looks like over time.
#
# This block also records WHEN information became known.
# That matters because future forecasting models cannot use information
# that was not available at the forecast origin.

# ----------------------------
# 5.1 Add launch-timing assumptions
# ----------------------------

new_assumptions_block_5 = [
    make_assumption(
        "COMP_002",
        "Therapy A launch month number",
        1,
        "month_index",
        "competition",
        "synthetic_design",
        SYNTHETIC_REFERENCE,
        "design_choice",
        False,
        "Therapy A is already available at the start of the simulation."
    ),
    make_assumption(
        "COMP_003",
        "Therapy B launch month number",
        19,
        "month_index",
        "competition",
        "synthetic_design",
        SYNTHETIC_REFERENCE,
        "design_choice",
        True,
        "Therapy B launches during the early market period."
    ),
    make_assumption(
        "COMP_004",
        "Therapy C launch month number",
        1,
        "month_index",
        "competition",
        "synthetic_design",
        SYNTHETIC_REFERENCE,
        "design_choice",
        False,
        "Therapy C is already available at the start of the simulation."
    ),
]

assumption_registry = pd.concat(
    [assumption_registry, pd.DataFrame(new_assumptions_block_5)],
    ignore_index=True
)

# Prevent duplicated assumption IDs if this block is accidentally rerun
assumption_registry = assumption_registry.drop_duplicates(
    subset=["assumption_id"],
    keep="last"
).reset_index(drop=True)

print(f"Updated registry: {len(assumption_registry)} assumptions")


# ----------------------------
# 5.2 Define therapy launch months
# ----------------------------

therapy_launches = pd.DataFrame({
    "therapy": ["Therapy A", "Therapy B", "Therapy C", "Therapy D"],
    "launch_month": [
        int(get_assumption_value("COMP_002")),
        int(get_assumption_value("COMP_003")),
        int(get_assumption_value("COMP_004")),
        int(get_assumption_value("COMP_001")),
    ],
    "launch_known_from_month": [
        1,    # A is already known at simulation start
        12,   # B launch is announced before launch
        1,    # C is already known at simulation start
        78,   # D launch is announced before launch
    ]
})

display(therapy_launches)


# ----------------------------
# 5.3 Define therapy eligibility
# ----------------------------

# This is the clinical/market logic:
#
# Therapy A: biomarker-positive, 1L only
# Therapy B: biomarker-positive, 1L and 2L
# Therapy C: biomarker-negative, 1L and 2L
# Therapy D: biomarker-positive, 1L and 2L, later entrant
#
# This creates explicit patient-segment overlap:
# A, B, and D overlap strongly in biomarker-positive patients.
# C mostly serves a different population.

eligibility_rows = []

for therapy in THERAPIES:
    for biomarker_status in BIOMARKERS:
        for line_of_therapy in LINES_OF_THERAPY:

            eligible = 0

            if therapy == "Therapy A":
                eligible = int(
                    biomarker_status == "Positive"
                    and line_of_therapy == "1L"
                )

            elif therapy == "Therapy B":
                eligible = int(
                    biomarker_status == "Positive"
                    and line_of_therapy in ["1L", "2L"]
                )

            elif therapy == "Therapy C":
                eligible = int(
                    biomarker_status == "Negative"
                    and line_of_therapy in ["1L", "2L"]
                )

            elif therapy == "Therapy D":
                eligible = int(
                    biomarker_status == "Positive"
                    and line_of_therapy in ["1L", "2L"]
                )

            eligibility_rows.append({
                "therapy": therapy,
                "biomarker_status": biomarker_status,
                "line_of_therapy": line_of_therapy,
                "eligible": eligible
            })

therapy_eligibility = pd.DataFrame(eligibility_rows)

print(f"Eligibility table created: {len(therapy_eligibility):,} rows")
display(therapy_eligibility)


# ----------------------------
# 5.4 Define event calendar
# ----------------------------

# The event calendar separates:
#
# announcement_month = when the market could know the event
# effective_month    = when the event affects the market
#
# This distinction is critical for point-in-time forecasting.

market_calendar_events = pd.DataFrame([
    {
        "event_id": "EVT_001",
        "event_name": "Therapy B launch",
        "event_type": "competitor_launch",
        "therapy": "Therapy B",
        "region": "ALL",
        "announcement_month": 12,
        "effective_month": 19,
        "information_category": "published_forward_information",
        "notes": "Therapy B launch becomes known seven months before launch."
    },
    {
        "event_id": "EVT_002",
        "event_name": "Therapy B East access expansion",
        "event_type": "access_change",
        "therapy": "Therapy B",
        "region": "East",
        "announcement_month": 55,
        "effective_month": 61,
        "information_category": "published_forward_information",
        "notes": "East access improves from 50% to 72%."
    },
    {
        "event_id": "EVT_003",
        "event_name": "Therapy D launch",
        "event_type": "competitor_launch",
        "therapy": "Therapy D",
        "region": "ALL",
        "announcement_month": 78,
        "effective_month": 85,
        "information_category": "published_forward_information",
        "notes": "Therapy D enters biomarker-positive first- and second-line segments."
    },
    {
        "event_id": "EVT_004",
        "event_name": "Therapy A persistence deterioration",
        "event_type": "persistence_change",
        "therapy": "Therapy A",
        "region": "ALL",
        "announcement_month": np.nan,
        "effective_month": 103,
        "information_category": "forbidden_until_observed",
        "notes": "Unanticipated persistence change. Forecasting models cannot know this before it appears in observed data."
    },
    {
        "event_id": "EVT_005",
        "event_name": "Epidemiology regime change",
        "event_type": "epidemiology_change",
        "therapy": "ALL",
        "region": "ALL",
        "announcement_month": 109,
        "effective_month": 115,
        "information_category": "published_forward_information",
        "notes": "Latent disease incidence increases by 12%."
    },
    {
        "event_id": "EVT_006",
        "event_name": "Therapy D limited supply",
        "event_type": "supply_constraint",
        "therapy": "Therapy D",
        "region": "East",
        "announcement_month": 124,
        "effective_month": 125,
        "information_category": "published_forward_information",
        "notes": "Limited supply event affecting observed sales but not latent demand."
    },
])

assert market_calendar_events["event_id"].is_unique

print(f"Event calendar created: {len(market_calendar_events):,} events")
display(market_calendar_events)


# ----------------------------
# 5.5 Define base access rates
# ----------------------------

# These are synthetic market-access assumptions.
# They create regional heterogeneity, which later helps us test whether
# patient/market information adds value beyond historical demand alone.

base_access_rates = pd.DataFrame([
    {"therapy": "Therapy A", "region": "North", "base_access_rate": 0.74},
    {"therapy": "Therapy A", "region": "South", "base_access_rate": 0.60},
    {"therapy": "Therapy A", "region": "East",  "base_access_rate": 0.55},
    {"therapy": "Therapy A", "region": "West",  "base_access_rate": 0.65},

    {"therapy": "Therapy B", "region": "North", "base_access_rate": 0.68},
    {"therapy": "Therapy B", "region": "South", "base_access_rate": 0.55},
    {"therapy": "Therapy B", "region": "East",  "base_access_rate": 0.50},
    {"therapy": "Therapy B", "region": "West",  "base_access_rate": 0.60},

    {"therapy": "Therapy C", "region": "North", "base_access_rate": 0.76},
    {"therapy": "Therapy C", "region": "South", "base_access_rate": 0.70},
    {"therapy": "Therapy C", "region": "East",  "base_access_rate": 0.66},
    {"therapy": "Therapy C", "region": "West",  "base_access_rate": 0.72},

    {"therapy": "Therapy D", "region": "North", "base_access_rate": 0.65},
    {"therapy": "Therapy D", "region": "South", "base_access_rate": 0.55},
    {"therapy": "Therapy D", "region": "East",  "base_access_rate": 0.50},
    {"therapy": "Therapy D", "region": "West",  "base_access_rate": 0.60},
])


# ----------------------------
# 5.6 Create monthly market-access table
# ----------------------------

market_access_monthly = (
    pd.MultiIndex
    .from_product([MONTHS, THERAPIES, REGIONS], names=["month", "therapy", "region"])
    .to_frame(index=False)
)

market_access_monthly["month_index"] = (
    (market_access_monthly["month"].dt.year - SIMULATION_START.year) * 12
    + (market_access_monthly["month"].dt.month - SIMULATION_START.month)
    + 1
)

market_access_monthly = market_access_monthly.merge(
    therapy_launches,
    on="therapy",
    how="left"
)

market_access_monthly = market_access_monthly.merge(
    base_access_rates,
    on=["therapy", "region"],
    how="left"
)

market_access_monthly["therapy_launched"] = (
    market_access_monthly["month_index"] >= market_access_monthly["launch_month"]
)

# Before launch, true access is zero.
# After launch, true access equals the base access rate, unless changed by events.
market_access_monthly["access_rate_truth"] = np.where(
    market_access_monthly["therapy_launched"],
    market_access_monthly["base_access_rate"],
    0.0
)

market_access_monthly["access_regime"] = np.where(
    market_access_monthly["therapy_launched"],
    "baseline",
    "not_launched"
)

# Access information is not known before a therapy launch is announced.
market_access_monthly["access_known_from_month"] = np.where(
    market_access_monthly["therapy_launched"],
    market_access_monthly["launch_known_from_month"],
    np.nan
)

# Apply Therapy B East access expansion:
# announced in month 55, effective in month 61.
b_east_access_expansion = (
    (market_access_monthly["therapy"] == "Therapy B")
    & (market_access_monthly["region"] == "East")
    & (market_access_monthly["month_index"] >= 61)
)

market_access_monthly.loc[b_east_access_expansion, "access_rate_truth"] = 0.72
market_access_monthly.loc[b_east_access_expansion, "access_regime"] = "east_access_expansion"
market_access_monthly.loc[b_east_access_expansion, "access_known_from_month"] = 55

market_access_monthly["access_information_category"] = np.where(
    market_access_monthly["therapy_launched"],
    "published_or_historical_information",
    "not_available_before_launch"
)

market_access_monthly = market_access_monthly[
    [
        "month",
        "month_index",
        "therapy",
        "region",
        "launch_month",
        "launch_known_from_month",
        "therapy_launched",
        "access_rate_truth",
        "access_regime",
        "access_known_from_month",
        "access_information_category",
    ]
].copy()

print(f"Monthly market-access table created: {len(market_access_monthly):,} rows")


# ----------------------------
# 5.7 Validation checks
# ----------------------------

assert len(market_access_monthly) == len(MONTHS) * len(THERAPIES) * len(REGIONS)

assert market_access_monthly["access_rate_truth"].between(0, 1).all()

# Therapy D should have zero access before launch.
assert (
    market_access_monthly.loc[
        (market_access_monthly["therapy"] == "Therapy D")
        & (market_access_monthly["month_index"] < 85),
        "access_rate_truth"
    ] == 0
).all()

# Therapy D launch should only be known from month 78.
assert (
    market_access_monthly.loc[
        (market_access_monthly["therapy"] == "Therapy D")
        & (market_access_monthly["month_index"] >= 85),
        "access_known_from_month"
    ] == 78
).all()

# Therapy B launch should only be known from month 12.
assert (
    market_access_monthly.loc[
        (market_access_monthly["therapy"] == "Therapy B")
        & (market_access_monthly["month_index"].between(19, 60)),
        "access_known_from_month"
    ] == 12
).all()

# Therapy B East expansion should be known from month 55 once effective.
assert (
    market_access_monthly.loc[
        (market_access_monthly["therapy"] == "Therapy B")
        & (market_access_monthly["region"] == "East")
        & (market_access_monthly["month_index"] >= 61),
        "access_known_from_month"
    ] == 55
).all()

# Before launch, access timing should not pretend the access rate was known.
assert market_access_monthly.loc[
    ~market_access_monthly["therapy_launched"],
    "access_known_from_month"
].isna().all()

print("Market-access validation checks passed.")


# ----------------------------
# 5.8 Display important rows
# ----------------------------

display(
    market_access_monthly[
        (market_access_monthly["therapy"] == "Therapy B")
        & (market_access_monthly["region"] == "East")
        & (market_access_monthly["month_index"].isin([54, 55, 60, 61, 62]))
    ]
)

display(
    market_access_monthly[
        (market_access_monthly["therapy"] == "Therapy D")
        & (market_access_monthly["region"] == "East")
        & (market_access_monthly["month_index"].isin([77, 78, 84, 85, 86, 125]))
    ]
)

In [ ]:
# ============================================================
# BLOCK 6 — Competitor Overlap and Treatment-Allocation Logic
# ============================================================

# Why this block exists:
# Competition should not be represented as "competitor_present = 1".
# A competitor matters only if it competes for the same patient segments.
#
# This block creates:
# 1. therapy market-position assumptions
# 2. patient-segment overlap between therapies
# 3. launch maturity logic
# 4. treatment-allocation weights
#
# The next block will use these rules to simulate starts, persistence,
# active patients, and patient-months.

# ----------------------------
# 6.1 Add treatment-allocation assumptions
# ----------------------------

new_assumptions_block_6 = [
    make_assumption(
        "MKT_001",
        "Outside option treatment weight",
        0.20,
        "relative_weight",
        "commercial",
        "synthetic_design",
        SYNTHETIC_REFERENCE,
        "design_choice",
        False,
        "Represents other treatments, no treatment, or treatments outside the modeled therapy set."
    ),
    make_assumption(
        "MKT_002",
        "Launch maturity speed",
        0.18,
        "curve_parameter",
        "commercial",
        "synthetic_design",
        SYNTHETIC_REFERENCE,
        "design_choice",
        False,
        "Controls how quickly a launched therapy approaches mature uptake."
    ),
    make_assumption(
        "MKT_003",
        "Launch maturity midpoint",
        12,
        "months_since_launch",
        "commercial",
        "synthetic_design",
        SYNTHETIC_REFERENCE,
        "design_choice",
        False,
        "A therapy reaches roughly half of its maturity effect around 12 months after launch."
    ),
]

assumption_registry = pd.concat(
    [assumption_registry, pd.DataFrame(new_assumptions_block_6)],
    ignore_index=True
)

# Prevent duplicate assumption IDs if this block is accidentally rerun
assumption_registry = assumption_registry.drop_duplicates(
    subset=["assumption_id"],
    keep="last"
).reset_index(drop=True)

print(f"Updated registry: {len(assumption_registry)} assumptions")


# ----------------------------
# 6.2 Define therapy positioning by segment
# ----------------------------

# This table is synthetic, but it has business meaning.
#
# A higher positioning_score means the therapy is more attractive
# within that eligible patient segment, before access and launch maturity.
#
# This is NOT the final market share.
# Final allocation will also depend on eligibility, launch status,
# access, and competing options.

therapy_positioning = pd.DataFrame([
    {
        "therapy": "Therapy A",
        "biomarker_status": "Positive",
        "line_of_therapy": "1L",
        "positioning_score": 1.00,
        "positioning_notes": "Established biomarker-positive first-line therapy."
    },
    {
        "therapy": "Therapy B",
        "biomarker_status": "Positive",
        "line_of_therapy": "1L",
        "positioning_score": 0.90,
        "positioning_notes": "Competes with Therapy A in first line."
    },
    {
        "therapy": "Therapy B",
        "biomarker_status": "Positive",
        "line_of_therapy": "2L",
        "positioning_score": 1.05,
        "positioning_notes": "Has meaningful second-line use."
    },
    {
        "therapy": "Therapy C",
        "biomarker_status": "Negative",
        "line_of_therapy": "1L",
        "positioning_score": 1.00,
        "positioning_notes": "Main modeled option for biomarker-negative first-line patients."
    },
    {
        "therapy": "Therapy C",
        "biomarker_status": "Negative",
        "line_of_therapy": "2L",
        "positioning_score": 0.95,
        "positioning_notes": "Continues to serve biomarker-negative second-line patients."
    },
    {
        "therapy": "Therapy D",
        "biomarker_status": "Positive",
        "line_of_therapy": "1L",
        "positioning_score": 1.20,
        "positioning_notes": "Later entrant with strong overlap against A/B."
    },
    {
        "therapy": "Therapy D",
        "biomarker_status": "Positive",
        "line_of_therapy": "2L",
        "positioning_score": 1.15,
        "positioning_notes": "Later entrant also competes in second line."
    },
])

print(f"Therapy positioning table created: {len(therapy_positioning):,} rows")
display(therapy_positioning)


# ----------------------------
# 6.3 Calculate patient-segment overlap
# ----------------------------

# Overlap answers:
# "How much of Therapy X's eligible population is also eligible for Therapy Y?"
#
# This is directional.
#
# Example:
# A overlaps strongly with B and D because A treats only Positive 1L,
# and B/D also treat Positive 1L.
#
# But B does not overlap 100% with A because B also treats Positive 2L,
# where A does not compete.

segment_weights = (
    patient_segment_inputs
    .loc[patient_segment_inputs["month_index"] == 1,
         ["region", "biomarker_status", "line_of_therapy",
          "new_first_line_diagnosis_inflow", "initial_active_patients"]]
    .copy()
)

segment_weights["segment_weight"] = (
    segment_weights["new_first_line_diagnosis_inflow"]
    + segment_weights["initial_active_patients"]
)

segment_weights = (
    segment_weights
    .groupby(["biomarker_status", "line_of_therapy"], as_index=False)
    ["segment_weight"]
    .sum()
)

eligibility_with_weights = therapy_eligibility.merge(
    segment_weights,
    on=["biomarker_status", "line_of_therapy"],
    how="left"
)

eligibility_with_weights["segment_weight"] = eligibility_with_weights["segment_weight"].fillna(0)

overlap_rows = []

for focal_therapy in THERAPIES:
    focal_segments = eligibility_with_weights[
        (eligibility_with_weights["therapy"] == focal_therapy)
        & (eligibility_with_weights["eligible"] == 1)
    ][["biomarker_status", "line_of_therapy", "segment_weight"]]

    focal_denominator = focal_segments["segment_weight"].sum()

    for competitor_therapy in THERAPIES:
        competitor_segments = therapy_eligibility[
            (therapy_eligibility["therapy"] == competitor_therapy)
            & (therapy_eligibility["eligible"] == 1)
        ][["biomarker_status", "line_of_therapy"]]

        shared_segments = focal_segments.merge(
            competitor_segments,
            on=["biomarker_status", "line_of_therapy"],
            how="inner"
        )

        shared_weight = shared_segments["segment_weight"].sum()

        overlap_score = (
            shared_weight / focal_denominator
            if focal_denominator > 0
            else 0
        )

        overlap_rows.append({
            "focal_therapy": focal_therapy,
            "competitor_therapy": competitor_therapy,
            "patient_segment_overlap": overlap_score
        })

therapy_overlap = pd.DataFrame(overlap_rows)

print("Directional patient-segment overlap created.")
display(
    therapy_overlap.pivot(
        index="focal_therapy",
        columns="competitor_therapy",
        values="patient_segment_overlap"
    ).round(2)
)


# ----------------------------
# 6.4 Create allocation-ready therapy options
# ----------------------------

# This table is still not the final patient-flow table.
# It tells us, for each month/region/segment/therapy:
#
# - Is the therapy clinically eligible?
# - Has it launched?
# - What is its access?
# - How mature is it after launch?
# - What is its relative allocation weight?

allocation_options = patient_segment_inputs.merge(
    therapy_eligibility,
    on=["biomarker_status", "line_of_therapy"],
    how="left"
)

allocation_options = allocation_options.merge(
    market_access_monthly,
    on=["month", "month_index", "therapy", "region"],
    how="left"
)

allocation_options = allocation_options.merge(
    therapy_positioning,
    on=["therapy", "biomarker_status", "line_of_therapy"],
    how="left"
)

allocation_options["positioning_score"] = allocation_options["positioning_score"].fillna(0)

allocation_options["months_since_launch"] = (
    allocation_options["month_index"] - allocation_options["launch_month"] + 1
)

allocation_options["months_since_launch"] = allocation_options["months_since_launch"].clip(lower=0)

launch_speed = get_assumption_value("MKT_002")
launch_midpoint = get_assumption_value("MKT_003")

allocation_options["launch_maturity"] = np.where(
    allocation_options["therapy_launched"],
    1 / (1 + np.exp(-launch_speed * (allocation_options["months_since_launch"] - launch_midpoint))),
    0
)

# Allocation weight:
# eligible × launched × access × launch maturity × positioning


allocation_options["therapy_allocation_weight"] = (
    allocation_options["eligible"]
    * allocation_options["therapy_launched"].astype(int)
    * allocation_options["access_rate_truth"]
    * allocation_options["launch_maturity"]
    * allocation_options["positioning_score"]
)


# ----------------------------
# 6.5 Convert weights into treatment shares
# ----------------------------

# We include an outside option so modeled therapies do not automatically
# capture 100% of patients.
#
# Outside option can represent:
# - another unmodeled treatment
# - no systemic treatment
# - delayed treatment
# - clinical-trial participation
#
# This is common in market models because the modeled product set is rarely
# the entire real treatment universe.

outside_option_weight = get_assumption_value("MKT_001")

share_denominator = (
    allocation_options
    .groupby(["month", "month_index", "region", "biomarker_status", "line_of_therapy"])
    ["therapy_allocation_weight"]
    .transform("sum")
    + outside_option_weight
)

allocation_options["treatment_share"] = (
    allocation_options["therapy_allocation_weight"] / share_denominator
)

allocation_options["outside_option_share"] = (
    outside_option_weight / share_denominator
)

# Keep only useful columns
allocation_options = allocation_options[
    [
        "month",
        "month_index",
        "region",
        "biomarker_status",
        "line_of_therapy",
        "therapy",
        "eligible",
        "therapy_launched",
        "launch_month",
        "launch_known_from_month",
        "access_rate_truth",
        "access_known_from_month",
        "access_regime",
        "positioning_score",
        "months_since_launch",
        "launch_maturity",
        "therapy_allocation_weight",
        "treatment_share",
        "outside_option_share",
    ]
].copy()

print(f"Allocation-options table created: {len(allocation_options):,} rows")


# ----------------------------
# 6.6 Validation checks
# ----------------------------

assert len(allocation_options) == len(patient_segment_inputs) * len(THERAPIES)

assert allocation_options["treatment_share"].between(0, 1).all()

assert allocation_options["outside_option_share"].between(0, 1).all()

# In every month-region-segment, modeled therapy shares + outside option
# should sum to 1.
share_check = (
    allocation_options
    .groupby(["month", "region", "biomarker_status", "line_of_therapy"], as_index=False)
    .agg(
        modeled_share_sum=("treatment_share", "sum"),
        outside_option_share=("outside_option_share", "first")
    )
)

share_check["total_share"] = (
    share_check["modeled_share_sum"]
    + share_check["outside_option_share"]
)

assert np.allclose(share_check["total_share"], 1.0)

# Therapy C should not compete in biomarker-positive segments.
assert (
    allocation_options.loc[
        (allocation_options["therapy"] == "Therapy C")
        & (allocation_options["biomarker_status"] == "Positive"),
        "treatment_share"
    ] == 0
).all()

# Therapy D should have zero share before launch.
assert (
    allocation_options.loc[
        (allocation_options["therapy"] == "Therapy D")
        & (allocation_options["month_index"] < 85),
        "treatment_share"
    ] == 0
).all()

print("Treatment-allocation validation checks passed.")


# ----------------------------
# 6.7 Display key examples
# ----------------------------

# Example 1:
# Positive 1L before Therapy D launch.
display(
    allocation_options[
        (allocation_options["region"] == "East")
        & (allocation_options["biomarker_status"] == "Positive")
        & (allocation_options["line_of_therapy"] == "1L")
        & (allocation_options["month_index"] == 84)
    ][
        [
            "month",
            "therapy",
            "eligible",
            "therapy_launched",
            "access_rate_truth",
            "launch_maturity",
            "positioning_score",
            "treatment_share",
            "outside_option_share",
        ]
    ].round(3)
)

# Example 2:
# Positive 1L after Therapy D launch.
display(
    allocation_options[
        (allocation_options["region"] == "East")
        & (allocation_options["biomarker_status"] == "Positive")
        & (allocation_options["line_of_therapy"] == "1L")
        & (allocation_options["month_index"] == 96)
    ][
        [
            "month",
            "therapy",
            "eligible",
            "therapy_launched",
            "access_rate_truth",
            "launch_maturity",
            "positioning_score",
            "treatment_share",
            "outside_option_share",
        ]
    ].round(3)
)

# Example 3:
# Negative 1L after Therapy D launch.
# Therapy D should not affect this segment because it is not eligible here.
display(
    allocation_options[
        (allocation_options["region"] == "East")
        & (allocation_options["biomarker_status"] == "Negative")
        & (allocation_options["line_of_therapy"] == "1L")
        & (allocation_options["month_index"] == 96)
    ][
        [
            "month",
            "therapy",
            "eligible",
            "therapy_launched",
            "access_rate_truth",
            "launch_maturity",
            "positioning_score",
            "treatment_share",
            "outside_option_share",
        ]
    ].round(3)
)

In [ ]:
# ============================================================
# BLOCK 7 — Patient-Flow Simulation
# ============================================================

# Why this block exists:
# Demand should not come directly from population or incidence.
# Demand should come from treated patients over time.
#
# This block simulates:
# - treatment starts
# - discontinuation
# - progression from 1L to 2L
# - active patients
# - patient-months
#
# This is the central patient-flow engine of Notebook 1.

# ----------------------------
# 7.1 Add patient-flow assumptions
# ----------------------------

new_assumptions_block_7 = [
    make_assumption(
        "CLN_006",
        "Monthly second-line initiation rate",
        0.650,
        "proportion",
        "clinical",
        "synthetic_design",
        SYNTHETIC_REFERENCE,
        "design_choice",
        False,
        "Share of second-line candidate pool starting a modeled or outside treatment."
    ),
    make_assumption(
        "PERS_005",
        "Therapy A monthly discontinuation rate after deterioration",
        0.110,
        "proportion",
        "persistence",
        "synthetic_design",
        SYNTHETIC_REFERENCE,
        "design_choice",
        True,
        "Therapy A persistence worsens from month 103 onward. This event is not known before it appears in observed data."
    ),
]

assumption_registry = pd.concat(
    [assumption_registry, pd.DataFrame(new_assumptions_block_7)],
    ignore_index=True
)

# Prevent duplicate assumption IDs if this block is accidentally rerun
assumption_registry = assumption_registry.drop_duplicates(
    subset=["assumption_id"],
    keep="last"
).reset_index(drop=True)

print(f"Updated registry: {len(assumption_registry)} assumptions")


# ----------------------------
# 7.2 Define persistence parameters
# ----------------------------

base_discontinuation_rates = {
    "Therapy A": get_assumption_value("PERS_001"),
    "Therapy B": get_assumption_value("PERS_002"),
    "Therapy C": get_assumption_value("PERS_003"),
    "Therapy D": get_assumption_value("PERS_004"),
}

therapy_a_deteriorated_discontinuation_rate = get_assumption_value("PERS_005")

first_line_initiation_rate = get_assumption_value("CLN_002")
second_line_initiation_rate = get_assumption_value("CLN_006")
first_line_progression_rate = get_assumption_value("CLN_003")

print("Base monthly discontinuation rates:")
display(pd.DataFrame(
    [{"therapy": k, "monthly_discontinuation_rate": v}
     for k, v in base_discontinuation_rates.items()]
))


# ----------------------------
# 7.3 Helper function for discontinuation rate
# ----------------------------

def get_monthly_discontinuation_rate(therapy, month_index):
    """
    Returns the monthly discontinuation rate for a therapy.

    Therapy A has a persistence deterioration from month 103 onward.
    This is a simulated truth, but it is not announced in advance.
    Forecasting models later must not know this before it appears in history.
    """

    if therapy == "Therapy A" and month_index >= 103:
        return therapy_a_deteriorated_discontinuation_rate

    return base_discontinuation_rates[therapy]


# ----------------------------
# 7.4 Simulate patient flow month by month
# ----------------------------

# We simulate sequentially because active patients depend on the previous month.
#
# previous_active stores active patients at the end of the previous month.
# previous_progression_pool stores patients who progressed from 1L last month
# and become 2L candidates this month.

patient_flow_rows = []

previous_active = {}
previous_progression_pool = {
    (region, biomarker_status): 0.0
    for region in REGIONS
    for biomarker_status in BIOMARKERS
}

month_indices = sorted(patient_segment_inputs["month_index"].unique())

for month_index in month_indices:

    current_month = MONTHS[month_index - 1]

    current_progression_pool = {
        (region, biomarker_status): 0.0
        for region in REGIONS
        for biomarker_status in BIOMARKERS
    }

    for region in REGIONS:
        for biomarker_status in BIOMARKERS:
            for line_of_therapy in LINES_OF_THERAPY:

                segment_row = patient_segment_inputs[
                    (patient_segment_inputs["month_index"] == month_index)
                    & (patient_segment_inputs["region"] == region)
                    & (patient_segment_inputs["biomarker_status"] == biomarker_status)
                    & (patient_segment_inputs["line_of_therapy"] == line_of_therapy)
                ].iloc[0]

                if line_of_therapy == "1L":
                    untreated_candidate_pool = segment_row["new_first_line_diagnosis_inflow"]
                    initiation_rate = first_line_initiation_rate

                else:
                    untreated_candidate_pool = previous_progression_pool[
                        (region, biomarker_status)
                    ]
                    initiation_rate = second_line_initiation_rate

                treated_start_pool = untreated_candidate_pool * initiation_rate

                option_rows = allocation_options[
                    (allocation_options["month_index"] == month_index)
                    & (allocation_options["region"] == region)
                    & (allocation_options["biomarker_status"] == biomarker_status)
                    & (allocation_options["line_of_therapy"] == line_of_therapy)
                ]

                for _, option in option_rows.iterrows():

                    therapy = option["therapy"]

                    treatment_share = option["treatment_share"]

                    new_starts = treated_start_pool * treatment_share

                    # Initial active patients exist only at simulation start.
                    # We allocate them across modeled therapies using the same treatment shares.
                    initial_active_patients = (
                        segment_row["initial_active_patients"] * treatment_share
                        if month_index == 1
                        else 0.0
                    )

                    patient_key = (
                        region,
                        biomarker_status,
                        line_of_therapy,
                        therapy
                    )

                    active_patients_start = previous_active.get(patient_key, 0.0)
                    active_patients_before_discontinuation = (
                        active_patients_start
                        + initial_active_patients
                        + new_starts
                    )

                    monthly_discontinuation_rate = get_monthly_discontinuation_rate(
                        therapy,
                        month_index
                    )

                    total_discontinuations = (
                        active_patients_before_discontinuation
                        * monthly_discontinuation_rate
                    )

                    # Progression is only from 1L to 2L.
                    # We keep progression as a subset of total discontinuation.
                    if line_of_therapy == "1L":
                        progression_rate = min(
                            first_line_progression_rate,
                            monthly_discontinuation_rate
                        )
                        progressions_to_second_line = (
                            active_patients_before_discontinuation
                            * progression_rate
                        )
                    else:
                        progressions_to_second_line = 0.0

                    non_progression_discontinuations = (
                        total_discontinuations
                        - progressions_to_second_line
                    )

                    active_patients_end = (
                        active_patients_before_discontinuation
                        - total_discontinuations
                    )

                    patient_months = active_patients_before_discontinuation

                    previous_active[patient_key] = active_patients_end

                    if line_of_therapy == "1L":
                        current_progression_pool[(region, biomarker_status)] += (
                            progressions_to_second_line
                        )

                    patient_flow_rows.append({
                        "month": current_month,
                        "month_index": month_index,
                        "region": region,
                        "biomarker_status": biomarker_status,
                        "line_of_therapy": line_of_therapy,
                        "therapy": therapy,

                        "untreated_candidate_pool": untreated_candidate_pool,
                        "initiation_rate": initiation_rate,
                        "treated_start_pool": treated_start_pool,
                        "treatment_share": treatment_share,

                        "new_starts": new_starts,
                        "initial_active_patients": initial_active_patients,
                        "active_patients_start": active_patients_start,
                        "active_patients_before_discontinuation": active_patients_before_discontinuation,

                        "monthly_discontinuation_rate": monthly_discontinuation_rate,
                        "total_discontinuations": total_discontinuations,
                        "progressions_to_second_line": progressions_to_second_line,
                        "non_progression_discontinuations": non_progression_discontinuations,

                        "active_patients_end": active_patients_end,
                        "patient_months": patient_months,
                    })

    previous_progression_pool = current_progression_pool.copy()

patient_flow = pd.DataFrame(patient_flow_rows)

print(f"Patient-flow table created: {len(patient_flow):,} rows")


# ----------------------------
# 7.5 Validation checks
# ----------------------------

expected_patient_flow_rows = (
    len(MONTHS)
    * len(REGIONS)
    * len(BIOMARKERS)
    * len(LINES_OF_THERAPY)
    * len(THERAPIES)
)

assert len(patient_flow) == expected_patient_flow_rows

numeric_cols_to_check = [
    "untreated_candidate_pool",
    "treated_start_pool",
    "new_starts",
    "initial_active_patients",
    "active_patients_start",
    "active_patients_before_discontinuation",
    "total_discontinuations",
    "progressions_to_second_line",
    "non_progression_discontinuations",
    "active_patients_end",
    "patient_months",
]

for col in numeric_cols_to_check:
    assert (patient_flow[col] >= -1e-8).all(), f"Negative values found in {col}"

# Ineligible therapy-segment combinations should have zero starts.
assert (
    patient_flow.merge(
        therapy_eligibility,
        on=["therapy", "biomarker_status", "line_of_therapy"],
        how="left"
    )
    .query("eligible == 0")["new_starts"]
    .abs()
    .max()
    < 1e-8
)

# Therapy D should have no starts before launch.
assert (
    patient_flow.loc[
        (patient_flow["therapy"] == "Therapy D")
        & (patient_flow["month_index"] < 85),
        "new_starts"
    ]
    .abs()
    .max()
    < 1e-8
)

# Progression should only occur in 1L.
assert (
    patient_flow.loc[
        patient_flow["line_of_therapy"] == "2L",
        "progressions_to_second_line"
    ]
    .abs()
    .max()
    < 1e-8
)

# Progression cannot exceed total discontinuation.
assert (
    patient_flow["progressions_to_second_line"]
    <= patient_flow["total_discontinuations"] + 1e-8
).all()

# Patient flow reconciliation:
# active_end = active_start + initial_active + starts - discontinuations
reconciliation_difference = (
    patient_flow["active_patients_end"]
    - (
        patient_flow["active_patients_start"]
        + patient_flow["initial_active_patients"]
        + patient_flow["new_starts"]
        - patient_flow["total_discontinuations"]
    )
)

assert reconciliation_difference.abs().max() < 1e-8

print("Patient-flow validation checks passed.")


# ----------------------------
# 7.6 Summarize patient flow

final_month_index = patient_flow["month_index"].max()

therapy_summary = (
    patient_flow
    .groupby("therapy", as_index=False)
    .agg(
        total_new_starts=("new_starts", "sum"),
        total_patient_months=("patient_months", "sum"),
    )
)

final_active_summary = (
    patient_flow
    .loc[patient_flow["month_index"] == final_month_index]
    .groupby("therapy", as_index=False)
    .agg(
        final_active_patients=("active_patients_end", "sum")
    )
)

therapy_summary = therapy_summary.merge(
    final_active_summary,
    on="therapy",
    how="left"
)

print("Therapy-level patient-flow summary:")
display(therapy_summary.round(1))


# ----------------------------
# 7.7 Show the Therapy D launch effect
# ----------------------------

# Positive 1L segment:
# Therapy D should begin taking starts after launch.
display(
    patient_flow[
        (patient_flow["region"] == "East")
        & (patient_flow["biomarker_status"] == "Positive")
        & (patient_flow["line_of_therapy"] == "1L")
        & (patient_flow["month_index"].isin([84, 85, 96]))
    ][
        [
            "month",
            "therapy",
            "new_starts",
            "active_patients_start",
            "total_discontinuations",
            "active_patients_end",
            "patient_months",
        ]
    ].round(2)
)


# ----------------------------
# 7.8 Show Therapy A persistence deterioration
# ----------------------------

# Around month 103, Therapy A's discontinuation rate increases.
# This is a simulated truth, but it was not announced in advance.

display(
    patient_flow[
        (patient_flow["therapy"] == "Therapy A")
        & (patient_flow["region"] == "North")
        & (patient_flow["biomarker_status"] == "Positive")
        & (patient_flow["line_of_therapy"] == "1L")
        & (patient_flow["month_index"].isin([102, 103, 104]))
    ][
        [
            "month",
            "month_index",
            "monthly_discontinuation_rate",
            "active_patients_before_discontinuation",
            "total_discontinuations",
            "active_patients_end",
        ]
    ].round(3)
)

In [ ]:
# ============================================================
# BLOCK 8 — Demand Generation, Latent Demand, and Observed Sales
# ============================================================

# Why this block exists:
# Patient flow tells us how many treated patient-months exist.
# Commercial demand converts those patient-months into product units.
#
# This block creates:
# - latent demand
# - limited supply logic
# - observed sales
#
# Key distinction:
# latent demand = what patients would use if supply were unconstrained
# observed sales = what is actually sold after supply constraints

# ----------------------------
# 8.1 Add supply assumptions
# ----------------------------

new_assumptions_block_8 = [
    make_assumption(
        "SUP_001",
        "Therapy D East limited-supply fill rate",
        0.700,
        "proportion",
        "supply",
        "synthetic_design",
        SYNTHETIC_REFERENCE,
        "design_choice",
        True,
        "During the limited-supply period, observed sales equal 70% of latent demand for Therapy D in East."
    ),
    make_assumption(
        "SUP_002",
        "Therapy D East limited-supply duration",
        6,
        "months",
        "supply",
        "synthetic_design",
        SYNTHETIC_REFERENCE,
        "design_choice",
        True,
        "Limited-supply period lasts six months after the effective month."
    ),
]

assumption_registry = pd.concat(
    [assumption_registry, pd.DataFrame(new_assumptions_block_8)],
    ignore_index=True
)

# Prevent duplicate assumption IDs if this block is accidentally rerun
assumption_registry = assumption_registry.drop_duplicates(
    subset=["assumption_id"],
    keep="last"
).reset_index(drop=True)

print(f"Updated registry: {len(assumption_registry)} assumptions")


# ----------------------------
# 8.2 Create utilization table
# ----------------------------

therapy_utilization = pd.DataFrame([
    {
        "therapy": "Therapy A",
        "units_per_patient_month": get_assumption_value("COMM_001")
    },
    {
        "therapy": "Therapy B",
        "units_per_patient_month": get_assumption_value("COMM_002")
    },
    {
        "therapy": "Therapy C",
        "units_per_patient_month": get_assumption_value("COMM_003")
    },
    {
        "therapy": "Therapy D",
        "units_per_patient_month": get_assumption_value("COMM_004")
    },
])

display(therapy_utilization)


# ----------------------------
# 8.3 Convert patient-months into latent demand
# ----------------------------

patient_demand = patient_flow.merge(
    therapy_utilization,
    on="therapy",
    how="left"
)

patient_demand["latent_demand_units"] = (
    patient_demand["patient_months"]
    * patient_demand["units_per_patient_month"]
)

# Default assumption:
# If there is no supply constraint, observed sales equal latent demand.
patient_demand["supply_fill_rate"] = 1.0
patient_demand["supply_regime"] = "unconstrained"
patient_demand["supply_known_from_month"] = np.nan


# ----------------------------
# 8.4 Apply limited supply event
# ----------------------------

supply_fill_rate = get_assumption_value("SUP_001")
supply_duration = int(get_assumption_value("SUP_002"))

supply_effective_month = int(
    market_calendar_events.loc[
        market_calendar_events["event_id"] == "EVT_006",
        "effective_month"
    ].iloc[0]
)

supply_announcement_month = int(
    market_calendar_events.loc[
        market_calendar_events["event_id"] == "EVT_006",
        "announcement_month"
    ].iloc[0]
)

supply_end_month = supply_effective_month + supply_duration - 1

limited_supply_mask = (
    (patient_demand["therapy"] == "Therapy D")
    & (patient_demand["region"] == "East")
    & (patient_demand["month_index"].between(supply_effective_month, supply_end_month))
)

patient_demand.loc[limited_supply_mask, "supply_fill_rate"] = supply_fill_rate
patient_demand.loc[limited_supply_mask, "supply_regime"] = "limited_supply"
patient_demand.loc[limited_supply_mask, "supply_known_from_month"] = supply_announcement_month


# ----------------------------
# 8.5 Create observed sales
# ----------------------------

patient_demand["observed_sales_units"] = (
    patient_demand["latent_demand_units"]
    * patient_demand["supply_fill_rate"]
)

patient_demand["unmet_demand_units"] = (
    patient_demand["latent_demand_units"]
    - patient_demand["observed_sales_units"]
)

print(f"Patient-level demand table created: {len(patient_demand):,} rows")


# ----------------------------
# 8.6 Aggregate to commercial demand table
# ----------------------------

# This is the main raw commercial table from Notebook 1.
# It has the recommended forecasting grain:
#
# therapy × region × month

commercial_demand = (
    patient_demand
    .groupby(["month", "month_index", "region", "therapy"], as_index=False)
    .agg(
        new_starts=("new_starts", "sum"),
        active_patients_end=("active_patients_end", "sum"),
        patient_months=("patient_months", "sum"),
        latent_demand_units=("latent_demand_units", "sum"),
        observed_sales_units=("observed_sales_units", "sum"),
        unmet_demand_units=("unmet_demand_units", "sum"),
        supply_constrained=("supply_regime", lambda x: int((x == "limited_supply").any())),
        supply_known_from_month=("supply_known_from_month", "min"),
    )
)

commercial_demand["supply_fill_rate"] = np.where(
    commercial_demand["latent_demand_units"] > 0,
    commercial_demand["observed_sales_units"] / commercial_demand["latent_demand_units"],
    1.0
)

commercial_demand["market_regime"] = "baseline"

commercial_demand.loc[
    commercial_demand["month_index"] < 19,
    "market_regime"
] = "early_market_before_b_launch"

commercial_demand.loc[
    commercial_demand["month_index"].between(19, 60),
    "market_regime"
] = "b_launch_growth"

commercial_demand.loc[
    commercial_demand["month_index"].between(61, 84),
    "market_regime"
] = "access_expansion"

commercial_demand.loc[
    commercial_demand["month_index"].between(85, 102),
    "market_regime"
] = "overlapping_competitor_launch"

commercial_demand.loc[
    commercial_demand["month_index"].between(103, 114),
    "market_regime"
] = "persistence_change"

commercial_demand.loc[
    commercial_demand["month_index"] >= 115,
    "market_regime"
] = "epidemiology_change"

commercial_demand.loc[
    commercial_demand["supply_constrained"] == 1,
    "market_regime"
] = "limited_supply"

print(f"Commercial demand table created: {len(commercial_demand):,} rows")


# ----------------------------
# 8.7 Validation checks
# ----------------------------

expected_commercial_rows = len(MONTHS) * len(REGIONS) * len(THERAPIES)

assert len(commercial_demand) == expected_commercial_rows

assert patient_demand["latent_demand_units"].ge(-1e-8).all()
assert patient_demand["observed_sales_units"].ge(-1e-8).all()
assert patient_demand["unmet_demand_units"].ge(-1e-8).all()

# Observed sales should never exceed latent demand.
assert (
    patient_demand["observed_sales_units"]
    <= patient_demand["latent_demand_units"] + 1e-8
).all()

# Demand should reconcile with patient-months and utilization.
demand_reconciliation = (
    patient_demand["latent_demand_units"]
    - (
        patient_demand["patient_months"]
        * patient_demand["units_per_patient_month"]
    )
)

assert demand_reconciliation.abs().max() < 1e-8

# Limited supply should only affect Therapy D in East.
limited_supply_rows = commercial_demand[commercial_demand["supply_constrained"] == 1]

assert (
    (limited_supply_rows["therapy"] == "Therapy D")
    & (limited_supply_rows["region"] == "East")
).all()

assert limited_supply_rows["month_index"].between(
    supply_effective_month,
    supply_end_month
).all()

# Unconstrained rows should have zero unmet demand.
assert (
    commercial_demand.loc[
        commercial_demand["supply_constrained"] == 0,
        "unmet_demand_units"
    ].abs().max()
    < 1e-8
)

print("Demand-generation validation checks passed.")


# ----------------------------
# 8.8 Display demand summary
# ----------------------------

therapy_demand_summary = (
    commercial_demand
    .groupby("therapy", as_index=False)
    .agg(
        total_latent_demand_units=("latent_demand_units", "sum"),
        total_observed_sales_units=("observed_sales_units", "sum"),
        total_unmet_demand_units=("unmet_demand_units", "sum"),
        final_month_observed_sales=("observed_sales_units", lambda x: x.iloc[-1])
    )
)

print("Therapy-level demand summary:")
display(therapy_demand_summary.round(1))


# ----------------------------
# 8.9 Show limited supply effect
# ----------------------------

display(
    commercial_demand[
        (commercial_demand["therapy"] == "Therapy D")
        & (commercial_demand["region"] == "East")
        & (commercial_demand["month_index"].between(123, 132))
    ][
        [
            "month",
            "month_index",
            "therapy",
            "region",
            "latent_demand_units",
            "observed_sales_units",
            "unmet_demand_units",
            "supply_fill_rate",
            "supply_constrained",
            "supply_known_from_month",
            "market_regime",
        ]
    ].round(2)
)

In [ ]:
# ============================================================
# BLOCK 9 — Governance, Data Dictionary, and Final Reconciliation
# ============================================================

# Why this block exists:
# Notebook 1 is not complete just because the code runs.
# We need to prove that the synthetic market is understandable,
# traceable, and internally consistent.
#
# This block creates:
# - raw table inventory
# - data dictionary
# - variable classification
# - forecast target definition
# - Forecast Information Set policy
# - final reconciliation checks

# ----------------------------
# 9.1 Raw synthetic table inventory
# ----------------------------

raw_synthetic_tables = {
    "assumption_registry": assumption_registry,
    "epidemiology_monthly": epidemiology_monthly,
    "patient_segment_inputs": patient_segment_inputs,
    "therapy_eligibility": therapy_eligibility,
    "market_calendar_events": market_calendar_events,
    "market_access_monthly": market_access_monthly,
    "therapy_positioning": therapy_positioning,
    "therapy_overlap": therapy_overlap,
    "allocation_options": allocation_options,
    "patient_flow": patient_flow,
    "patient_demand": patient_demand,
    "commercial_demand": commercial_demand,
}

table_inventory = pd.DataFrame([
    {
        "table_name": table_name,
        "rows": len(table),
        "columns": table.shape[1],
    }
    for table_name, table in raw_synthetic_tables.items()
])

print("Raw synthetic table inventory:")
display(table_inventory)


# ----------------------------
# 9.2 Define table grains and purpose
# ----------------------------

data_dictionary = pd.DataFrame([
    {
        "table_name": "assumption_registry",
        "grain": "one row per assumption",
        "purpose": "Stores assumption values, source type, confidence, and scenario eligibility."
    },
    {
        "table_name": "epidemiology_monthly",
        "grain": "region × month",
        "purpose": "Creates population, latent disease incidence, diagnosed NSCLC, and metastatic diagnoses."
    },
    {
        "table_name": "patient_segment_inputs",
        "grain": "region × month × biomarker × line of therapy",
        "purpose": "Splits metastatic patients into biomarker and treatment-line segments."
    },
    {
        "table_name": "therapy_eligibility",
        "grain": "therapy × biomarker × line of therapy",
        "purpose": "Defines which therapies are clinically eligible for which patient segments."
    },
    {
        "table_name": "market_calendar_events",
        "grain": "one row per market event",
        "purpose": "Records announcement timing, effective timing, and information category."
    },
    {
        "table_name": "market_access_monthly",
        "grain": "therapy × region × month",
        "purpose": "Defines access over time and when access information became known."
    },
    {
        "table_name": "therapy_positioning",
        "grain": "therapy × biomarker × line of therapy",
        "purpose": "Defines synthetic therapy attractiveness within eligible segments."
    },
    {
        "table_name": "therapy_overlap",
        "grain": "focal therapy × competitor therapy",
        "purpose": "Measures patient-segment overlap between therapies."
    },
    {
        "table_name": "allocation_options",
        "grain": "region × month × biomarker × line of therapy × therapy",
        "purpose": "Converts eligibility, access, launch maturity, and positioning into treatment shares."
    },
    {
        "table_name": "patient_flow",
        "grain": "region × month × biomarker × line of therapy × therapy",
        "purpose": "Simulates starts, discontinuation, progression, active patients, and patient-months."
    },
    {
        "table_name": "patient_demand",
        "grain": "region × month × biomarker × line of therapy × therapy",
        "purpose": "Converts patient-months into latent demand and observed sales."
    },
    {
        "table_name": "commercial_demand",
        "grain": "therapy × region × month",
        "purpose": "Final raw demand table used later for forecasting."
    },
])

print("Data dictionary:")
display(data_dictionary)


# ----------------------------
# 9.3 Classify important variables
# ----------------------------

variable_classification = pd.DataFrame([
    {
        "variable": "population",
        "classification": "SIMULATED TRUTH",
        "explanation": "Generated from starting regional population and population growth assumption."
    },
    {
        "variable": "annual_latent_disease_incidence_per_100k_truth",
        "classification": "SIMULATED TRUTH",
        "explanation": "Latent disease rate used to generate incident disease cases."
    },
    {
        "variable": "diagnosis_rate_truth",
        "classification": "ASSUMED PARAMETER",
        "explanation": "Synthetic assumption used to convert latent disease into diagnosed NSCLC."
    },
    {
        "variable": "new_diagnosed_nsclc_cases",
        "classification": "DERIVED VARIABLE",
        "explanation": "Incident disease cases multiplied by diagnosis rate."
    },
    {
        "variable": "new_metastatic_diagnoses",
        "classification": "DERIVED VARIABLE",
        "explanation": "Diagnosed NSCLC cases multiplied by metastatic-stage share."
    },
    {
        "variable": "biomarker_status",
        "classification": "DERIVED VARIABLE",
        "explanation": "Patient segment assigned using biomarker prevalence assumption."
    },
    {
        "variable": "eligible",
        "classification": "ASSUMED PARAMETER",
        "explanation": "Clinical eligibility rule defined by synthetic therapy design."
    },
    {
        "variable": "access_rate_truth",
        "classification": "SIMULATED TRUTH",
        "explanation": "True access rate in the synthetic world."
    },
    {
        "variable": "access_known_from_month",
        "classification": "FORECAST INFORMATION SET CONTROL",
        "explanation": "Prevents models from using access information before it was knowable."
    },
    {
        "variable": "treatment_share",
        "classification": "DERIVED VARIABLE",
        "explanation": "Calculated from eligibility, launch status, access, maturity, and positioning."
    },
    {
        "variable": "new_starts",
        "classification": "DERIVED VARIABLE",
        "explanation": "Candidate pool multiplied by initiation rate and treatment share."
    },
    {
        "variable": "active_patients_end",
        "classification": "DERIVED VARIABLE",
        "explanation": "Previous active patients plus starts minus discontinuations."
    },
    {
        "variable": "patient_months",
        "classification": "DERIVED VARIABLE",
        "explanation": "Approximation of treated exposure during the month."
    },
    {
        "variable": "latent_demand_units",
        "classification": "DERIVED VARIABLE",
        "explanation": "Patient-months multiplied by units per patient-month."
    },
    {
        "variable": "observed_sales_units",
        "classification": "OBSERVED SYNTHETIC OUTPUT",
        "explanation": "Final observed sales after any supply constraint."
    },
])

print("Variable classification:")
display(variable_classification)


# ----------------------------
# 9.4 Forecast target definition
# ----------------------------

forecast_target_definition = pd.DataFrame([
    {
        "target_name": "observed_sales_units",
        "forecasting_grain": "therapy × region × month",
        "reason": "This is the commercial outcome a forecaster would observe and forecast.",
        "important_note": "latent_demand_units is preserved for explanation, but observed_sales_units is the target."
    }
])

print("Forecast target definition:")
display(forecast_target_definition)


# ----------------------------
# 9.5 Forecast Information Set policy
# ----------------------------

forecast_information_set_policy = pd.DataFrame([
    {
        "information_category": "observed_historical_information",
        "allowed_use": "Allowed if observed on or before forecast origin.",
        "example": "Observed sales through month tau."
    },
    {
        "information_category": "published_forward_information",
        "allowed_use": "Allowed only if announcement month is on or before forecast origin.",
        "example": "Therapy D launch announced in month 78, effective in month 85."
    },
    {
        "information_category": "planning_assumption",
        "allowed_use": "Allowed for scenario analysis, not for ordinary accuracy testing unless explicitly defined at forecast origin.",
        "example": "User-defined downside scenario where access falls to 60%."
    },
    {
        "information_category": "forbidden_information",
        "allowed_use": "Not allowed before it becomes observable.",
        "example": "Unannounced Therapy A persistence deterioration before its effects appear in observed data."
    },
])

print("Forecast Information Set policy:")
display(forecast_information_set_policy)


# ----------------------------
# 9.6 Correct therapy-level final summary
# ----------------------------

final_month_index = commercial_demand["month_index"].max()

therapy_demand_summary_corrected = (
    commercial_demand
    .groupby("therapy", as_index=False)
    .agg(
        total_latent_demand_units=("latent_demand_units", "sum"),
        total_observed_sales_units=("observed_sales_units", "sum"),
        total_unmet_demand_units=("unmet_demand_units", "sum"),
    )
)

final_month_sales = (
    commercial_demand
    .loc[commercial_demand["month_index"] == final_month_index]
    .groupby("therapy", as_index=False)
    .agg(
        final_month_observed_sales=("observed_sales_units", "sum")
    )
)

therapy_demand_summary_corrected = therapy_demand_summary_corrected.merge(
    final_month_sales,
    on="therapy",
    how="left"
)

print("Corrected therapy-level demand summary:")
display(therapy_demand_summary_corrected.round(1))


# ----------------------------
# 9.7 Market regime summary
# ----------------------------

market_regime_summary = (
    commercial_demand
    .groupby("market_regime", as_index=False)
    .agg(
        rows=("observed_sales_units", "size"),
        total_observed_sales_units=("observed_sales_units", "sum")
    )
    .sort_values("rows")
)

print("Market regime summary:")
display(market_regime_summary.round(1))


# ----------------------------
# 9.8 Final reconciliation checks
# ----------------------------

# Assumptions and event governance
assert assumption_registry["assumption_id"].is_unique
assert market_calendar_events["event_id"].is_unique

# Final commercial-demand grain must be unique.
assert (
    commercial_demand
    .groupby(["therapy", "region", "month"])
    .size()
    .max()
    == 1
)

# Forecast target cannot be missing.
assert commercial_demand["observed_sales_units"].notna().all()

# No negative demand.
assert commercial_demand["latent_demand_units"].ge(-1e-8).all()
assert commercial_demand["observed_sales_units"].ge(-1e-8).all()
assert commercial_demand["unmet_demand_units"].ge(-1e-8).all()

# Observed sales cannot exceed latent demand.
assert (
    commercial_demand["observed_sales_units"]
    <= commercial_demand["latent_demand_units"] + 1e-8
).all()

# Supply-constrained rows must have unmet demand.
assert (
    commercial_demand.loc[
        commercial_demand["supply_constrained"] == 1,
        "unmet_demand_units"
    ]
    > 0
).all()

# Unconstrained rows should not have unmet demand.
assert (
    commercial_demand.loc[
        commercial_demand["supply_constrained"] == 0,
        "unmet_demand_units"
    ].abs().max()
    < 1e-8
)

# Patient-months should reconcile into latent demand.
patient_month_reconciliation = (
    patient_demand["latent_demand_units"]
    - patient_demand["patient_months"] * patient_demand["units_per_patient_month"]
)

assert patient_month_reconciliation.abs().max() < 1e-8

# Therapy D should not create demand before launch.
assert (
    commercial_demand.loc[
        (commercial_demand["therapy"] == "Therapy D")
        & (commercial_demand["month_index"] < 85),
        "observed_sales_units"
    ].abs().max()
    < 1e-8
)

# Therapy C should not be directly affected by Therapy D eligibility,
# because Therapy C serves biomarker-negative patients.
assert (
    therapy_overlap.loc[
        (therapy_overlap["focal_therapy"] == "Therapy C")
        & (therapy_overlap["competitor_therapy"] == "Therapy D"),
        "patient_segment_overlap"
    ].iloc[0]
    == 0
)




In [ ]:
# BLOCK 10 — Senior Review Gate for Notebook 1
# We are checking whether the synthetic market is strong enough to freeze before moving to Notebook 2.

senior_review_gate = pd.DataFrame([
    {
        "review_area": "Business objective",
        "assessment": "PASS",
        "review_comment": "The notebook creates a patient-based oncology demand world, not a generic sales dataset."
    },
    {
        "review_area": "Patient-flow logic",
        "assessment": "PASS",
        "review_comment": "Demand is generated through incidence, diagnosis, segmentation, eligibility, access, starts, persistence, active patients, and patient-months."
    },
    {
        "review_area": "Therapy eligibility",
        "assessment": "PASS",
        "review_comment": "Therapy A/B/D serve biomarker-positive patients; Therapy C serves biomarker-negative patients."
    },
    {
        "review_area": "Competitor overlap",
        "assessment": "PASS",
        "review_comment": "Competition is based on shared eligible patient segments, not a simple competitor flag."
    },
    {
        "review_area": "Persistence",
        "assessment": "PASS",
        "review_comment": "Active patients depend on continuation and discontinuation, and 1L progression feeds the 2L candidate pool."
    },
    {
        "review_area": "Latent demand vs observed sales",
        "assessment": "PASS",
        "review_comment": "Latent demand is preserved separately from observed sales, with a limited-supply event reducing observed sales."
    },
    {
        "review_area": "Forecasting target",
        "assessment": "PASS",
        "review_comment": "The final target is observed_sales_units at therapy × region × month grain."
    },
    {
        "review_area": "Forecast Information Set",
        "assessment": "PASS",
        "review_comment": "Important future information has known-from timing, which supports point-in-time governance in Notebook 2."
    },
    {
        "review_area": "Evidence use",
        "assessment": "PASS WITH LIMITATION",
        "review_comment": "Population growth, NSCLC incidence, prevalence, and metastatic-stage share are evidence-informed. Biomarker, access, persistence, and share assumptions remain synthetic design choices."
    },
    {
        "review_area": "External validity",
        "assessment": "LIMITATION",
        "review_comment": "This is a synthetic US-proxy oncology market. It is suitable for a controlled forecasting experiment, not for real commercial inference."
    },
    {
        "review_area": "Leakage risk",
        "assessment": "CONTROLLED FOR NOTEBOOK 1",
        "review_comment": "Raw truth tables exist, but Notebook 2 must enforce as-of feature creation. Models must not directly consume future truth."
    },
    {
        "review_area": "Reproducibility",
        "assessment": "PASS",
        "review_comment": "The simulation uses a fixed random seed and explicit assumptions."
    },
    {
        "review_area": "Client POC value",
        "assessment": "PASS",
        "review_comment": "The notebook supports a pharma planning story: eligible population, access, competition, persistence, supply, and demand."
    },
    {
        "review_area": "Hiring value",
        "assessment": "PASS",
        "review_comment": "The design shows simulation, forecasting setup, healthcare domain logic, data governance, and decision-support thinking."
    },
])

display(senior_review_gate)


# ----------------------------
# Key strengths
# ----------------------------

notebook_1_strengths = [
    "Demand is generated from patient flow, not directly from sales noise.",
    "The synthetic market can test when patient and market information matters.",
    "Therapy competition depends on patient-segment overlap.",
    "Latent demand and observed sales are separated.",
    "Forecast Information Set timing is built into the raw data design.",
    "The final forecasting grain is clear: therapy × region × month.",
]

print("What is strong:")
for item in notebook_1_strengths:
    print(f"- {item}")


# ----------------------------
# Known limitations
# ----------------------------

notebook_1_limitations = [
    "This is a synthetic case study, not a clinically validated NSCLC model.",
    "Some assumptions are evidence-informed, but several are synthetic design choices.",
    "Treatment allocation is simplified and does not represent physician-level decision-making.",
    "Patient-months are approximated at monthly granularity.",
    "Regions are fictional payer regions, not real US geographies.",
    "The model uses aggregated patient segments, not patient-level survival or claims history.",
]

print("\nWhat is weak or limited:")
for item in notebook_1_limitations:
    print(f"- {item}")


# ----------------------------
# What reviewers may challenge
# ----------------------------

reviewer_challenges = pd.DataFrame([
    {
        "reviewer": "Senior Data Scientist",
        "likely_challenge": "How do you prove the hybrid model is not being designed to win?",
        "defense": "The simulation allows history-driven, patient-informed, and hybrid models to succeed or fail by regime. We will evaluate with rolling-origin validation and Forecast Value Add."
    },
    {
        "reviewer": "Pharma client",
        "likely_challenge": "Are these assumptions real?",
        "defense": "Some are public-data-informed proxies, but this remains a synthetic POC. Assumption source_type clearly separates evidence-backed values from synthetic design choices."
    },
    {
        "reviewer": "General DS interviewer",
        "likely_challenge": "Why not just forecast sales directly?",
        "defense": "Because the research question is about information value. We need a controlled data-generating process where patient and market mechanisms actually generate demand."
    },
    {
        "reviewer": "Healthcare DS hiring manager",
        "likely_challenge": "Why aggregated segments instead of patient-level data?",
        "defense": "The business problem is demand planning at therapy-region-month level. Aggregated flow is proportional, explainable, and closer to many forecasting workflows."
    },
    {
        "reviewer": "Forecasting reviewer",
        "likely_challenge": "How will you prevent leakage?",
        "defense": "Notebook 2 will create point-in-time features using observed historical data, published forward information, and scenario assumptions only."
    },
])

print("\nReviewer challenge table:")
display(reviewer_challenges)


# ----------------------------
# Freeze decision
# ----------------------------

critical_items = [
    "No negative patient counts",
    "Eligibility rules reconcile",
    "Treatment shares reconcile",
    "Patient flows reconcile",
    "Demand reconciles with patient-months and utilization",
    "Observed sales do not exceed latent demand",
    "Competitor overlap behaves logically",
    "Forecasting target and grain are defined",
    "Information timing is captured for future feature governance",
]

print("\nCritical completion items:")
for item in critical_items:
    print(f"- {item}")



In [ ]:
# Save outputs
# ----------------------------
assumption_registry.to_csv("assumption_registry.csv", index=False)
epidemiology_monthly.to_csv("epidemiology_monthly.csv", index=False)
patient_segment_inputs.to_csv("patient_segment_inputs.csv", index=False)
therapy_eligibility.to_csv("therapy_eligibility.csv", index=False)
market_calendar_events.to_csv("market_calendar_events.csv", index=False)
market_access_monthly.to_csv("market_access_monthly.csv", index=False)
therapy_overlap.to_csv("therapy_overlap.csv", index=False)
patient_flow.to_csv("patient_flow.csv", index=False)
commercial_demand.to_csv("commercial_demand.csv", index=False)
data_dictionary.to_csv("data_dictionary.csv", index=False)
variable_classification.to_csv("variable_classification.csv", index=False)
forecast_information_set_policy.to_csv("forecast_information_set_policy.csv", index=False)
